# 02 — Modelo KNN

| Modelo | Justificación  |
|--------|---------------------------|
| **KNN** | Se utiliza como primer modelo basado en similitud para clasificar tickets según casos históricos parecidos. |

## Modelo predictivo para anticipar incumplimientos de SLA en HR Operations  
## (Predictive Model to Anticipate SLA Breaches in HR Operations)

El objetivo de este notebook es aplicar el primer modelo de Machine Learning del proyecto: KNN.

KNN significa K-Nearest Neighbors, o K vecinos más cercanos. Es un algoritmo de clasificación supervisada que predice la clase de un nuevo caso comparándolo con los casos históricos más similares.

En este proyecto, el modelo intentará predecir si un ticket incumplirá o no su SLA utilizando variables disponibles antes del cierre del caso.

In [25]:
# Importar librerías necesarias
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [26]:
# carga el dataset final:
ruta_datos = "../data/processed/tickets_hr_sla_model_ready.csv"

df = pd.read_csv(ruta_datos)

df.head()

,Customer Age,Product Purchased,Ticket Type,Ticket Subject,Ticket Priority,Ticket Channel,incumplio_sla
0,48,Dell XPS,Technical issue,Network problem,Low,Social media,0
1,27,Microsoft Office,Billing inquiry,Account access,Low,Social media,0
2,67,Autodesk AutoCAD,Billing inquiry,Data loss,Low,Email,0
3,48,Nintendo Switch,Cancellation request,Data loss,High,Phone,0
4,51,Microsoft Xbox Controller,Product inquiry,Software bug,High,Chat,1


In [27]:
# Ver el tamaño:
df.shape

(2769, 7)

In [28]:
# Ver columnas:
df.columns

Index(['Customer Age', 'Product Purchased', 'Ticket Type', 'Ticket Subject',
       'Ticket Priority', 'Ticket Channel', 'incumplio_sla'],
      dtype='object')

## Separación de variables predictoras y variable objetivo

El dataset final ya contiene únicamente variables seguras para modelado.

La variable objetivo es `incumplio_sla`.

- `0`: el ticket cumplió el SLA.
- `1`: el ticket incumplió el SLA.

In [29]:
X = df.drop(columns="incumplio_sla")
y = df["incumplio_sla"]

X.head()

,Customer Age,Product Purchased,Ticket Type,Ticket Subject,Ticket Priority,Ticket Channel
0,48,Dell XPS,Technical issue,Network problem,Low,Social media
1,27,Microsoft Office,Billing inquiry,Account access,Low,Social media
2,67,Autodesk AutoCAD,Billing inquiry,Data loss,Low,Email
3,48,Nintendo Switch,Cancellation request,Data loss,High,Phone
4,51,Microsoft Xbox Controller,Product inquiry,Software bug,High,Chat


In [30]:
y.value_counts(normalize=True).mul(100).round(2)

incumplio_sla
0    61.79
1    38.21
Name: proportion, dtype: float64

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (2215, 6)
X_test: (554, 6)
y_train: (2215,)
y_test: (554,)


## Preprocesamiento para KNN

KNN calcula distancias entre observaciones. Por este motivo, las variables numéricas deben reescalarse.

Además, las variables categóricas deben convertirse a formato numérico mediante One-Hot Encoding.

Se aplicará:

- `StandardScaler` para variables numéricas.
- `OneHotEncoder` para variables categóricas.

In [32]:
columnas_numericas = ["Customer Age"]

columnas_categoricas = [
    "Product Purchased",
    "Ticket Type",
    "Ticket Subject",
    "Ticket Priority",
    "Ticket Channel"
]

preprocesamiento = ColumnTransformer(
    transformers=[
        ("numericas", StandardScaler(), columnas_numericas),
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas)
    ]
)

In [33]:
modelo_knn = Pipeline(
    steps=[
        ("preprocesamiento", preprocesamiento),
        ("modelo", KNeighborsClassifier(n_neighbors=5))
    ]
)

modelo_knn

Pipeline(steps=[('preprocesamiento',
                 ColumnTransformer(transformers=[('numericas', StandardScaler(),
                                                  ['Customer Age']),
                                                 ('categoricas',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Product Purchased',
                                                   'Ticket Type',
                                                   'Ticket Subject',
                                                   'Ticket Priority',
                                                   'Ticket Channel'])])),
                ('modelo', KNeighborsClassifier())])

In [34]:
modelo_knn.fit(X_train, y_train)

Pipeline(steps=[('preprocesamiento',
                 ColumnTransformer(transformers=[('numericas', StandardScaler(),
                                                  ['Customer Age']),
                                                 ('categoricas',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Product Purchased',
                                                   'Ticket Type',
                                                   'Ticket Subject',
                                                   'Ticket Priority',
                                                   'Ticket Channel'])])),
                ('modelo', KNeighborsClassifier())])

In [35]:
y_pred_knn = modelo_knn.predict(X_test)

y_pred_knn[:20]

array([1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0])

In [36]:
accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

print("Accuracy:", round(accuracy_knn, 4))
print("Precision clase 1:", round(precision_knn, 4))
print("Recall clase 1:", round(recall_knn, 4))
print("F1-score clase 1:", round(f1_knn, 4))

Accuracy: 0.713
Precision clase 1: 0.6432
Recall clase 1: 0.5613
F1-score clase 1: 0.5995


In [37]:
matriz_knn = confusion_matrix(y_test, y_pred_knn)

matriz_knn

array([[276,  66],
       [ 93, 119]])

## Interpretación de la Matriz de Confusión

La matriz se lee así:

| Elemento | Valor | Interpretación |
|----------|-------|----------------|
| TN | 276 | Casos que cumplieron SLA y el modelo predijo correctamente que cumplirían |
| FP | 66 | Casos que cumplían SLA, pero el modelo los marcó como riesgo |
| FN | 93 | Casos que incumplieron SLA, pero el modelo no los detectó |
| TP | 119 | Casos que incumplieron SLA y el modelo sí detectó |

In [38]:
print(classification_report(y_test, y_pred_knn))

              precision    recall  f1-score   support

           0       0.75      0.81      0.78       342
           1       0.64      0.56      0.60       212

    accuracy                           0.71       554
   macro avg       0.70      0.68      0.69       554
weighted avg       0.71      0.71      0.71       554



In [39]:
resultados_knn = pd.DataFrame({
    "modelo": ["KNN"],
    "accuracy": [accuracy_knn],
    "precision_clase_1": [precision_knn],
    "recall_clase_1": [recall_knn],
    "f1_clase_1": [f1_knn]
})

resultados_knn

,modelo,accuracy,precision_clase_1,recall_clase_1,f1_clase_1
0,KNN,0.712996,0.643243,0.561321,0.599496


El modelo KNN consigue una accuracy del 71%, lo que significa que acierta aproximadamente 7 de cada 10 predicciones.

Pero lo más importante es el: recall_clase_1 = 0.56

Eso significa:

De todos los tickets que realmente incumplieron SLA, KNN detectó correctamente el 56%.

Es un resultado aceptable como primer modelo, pero todavía mejorable.

## Interpretación inicial del modelo KNN

KNN fue utilizado como primer modelo de clasificación supervisada para establecer una referencia inicial basada en similitud entre tickets.

El modelo compara cada nuevo caso con los casos históricos más parecidos y clasifica el ticket según el comportamiento de sus vecinos más cercanos.

En este proyecto, la métrica más importante es el `recall` de la clase `1`, porque representa la capacidad del modelo para detectar tickets que realmente incumplen SLA.

Este modelo servirá como punto de comparación frente a otros algoritmos como Logistic Regression, Decision Tree, Random Forest y Gradient Boosting.

## Interpretación de resultados del modelo KNN

El modelo KNN obtuvo una accuracy general de 0,71, lo que indica que clasificó correctamente aproximadamente el 71% de los tickets del conjunto de test.

Sin embargo, en este proyecto la métrica más importante no es únicamente la accuracy, sino el recall de la clase `1`, ya que esta clase representa los tickets que incumplen SLA.

Para la clase `1`, el modelo obtuvo:

| Métrica | Valor |
|---|---:|
| Precision | 0,64 |
| Recall | 0,56 |
| F1-score | 0,60 |

Esto significa que, de todos los tickets que realmente incumplieron SLA, el modelo detectó correctamente el 56%.

La matriz de confusión muestra que el modelo identificó correctamente 119 incumplimientos de SLA, pero no detectó 93 casos que también incumplieron. Estos falsos negativos son especialmente relevantes desde el punto de vista de negocio, porque representan tickets de riesgo que no habrían sido priorizados a tiempo.

Por tanto, KNN funciona como un primer modelo de referencia, pero no debería considerarse todavía como el modelo final. Su rendimiento es aceptable como punto de partida, aunque será necesario compararlo con otros modelos como Logistic Regression, Decision Tree, Random Forest y Gradient Boosting.

## Conclusión del notebook

En este notebook se aplicó el primer modelo de Machine Learning del proyecto: KNN.

El modelo permitió establecer una primera referencia predictiva para anticipar incumplimientos de SLA en tickets operativos.

Aunque el resultado general fue aceptable, con una accuracy del 71%, el recall de la clase `1` fue de 0,56. Esto indica que el modelo todavía deja sin detectar una parte relevante de los tickets que realmente incumplen SLA.

Desde una perspectiva de negocio, este resultado confirma que KNN puede servir como baseline inicial, pero no es suficiente para tomarlo como modelo final.

El siguiente paso será comparar KNN con otros modelos de clasificación supervisada para identificar cuál ofrece mejor capacidad de detección de incumplimientos.